# 11 — Cross-Session Replication

**Purpose.** Replicate the four headline analyses from session 1055240613 (rounds 1–3) across a small matched cohort of VBN sessions. Target: determine which findings generalize vs which were single-session quirks.

**Cohort.** Three familiar-G-image sessions selected by coverage match (≥8 of the critical areas: LGd, VISp, VISl, VISal, VISrl, VISpm, VISam, CA1, DG-\*, ProS, SCig, MRN):
- **1055240613** — primary (mouse 533539)
- **1067588044** — mouse 544836, 12/12 critical areas
- **1115086689** — mouse 574078, 12/12 critical areas

All three are `EPHYS_1_images_G_3uL_reward`, experience level `Familiar`, 6 probes, wild-type genotype.

**Analyses run per session** (NWB-only, video-independent):
- **A2** — active-vs-passive per-unit MI, arousal-matched (`pupil` + `running` IQR of active block)
- **B2** — change amplification: old baseline / strict (rl≥250ms) / miss-only, all vs adaptation-matched (flashes_since_change=1)
- **D2** — encoding R² per area (running + 4 pupil features, 8-basis raised cosine lag expansion, ridge with gap_bins=40)
- **H** — within-area noise correlations by state (flash-residualized 25ms counts, paired Wilcoxon active vs passive)

Outputs land under `outputs/cross_session/<session_id>/` and get aggregated into `reports/cross_session/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts' / 'analysis'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _per_session_pipeline import run_all

SESSIONS = [1055240613, 1067588044, 1115086689]
CROSS_REPORT_DIR = ROOT / 'reports' / 'cross_session'
CROSS_REPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Sessions:', SESSIONS)
print('Cross-session reports →', CROSS_REPORT_DIR)

## Step 1 — Run the full pipeline per session

Each session auto-extracts from the AllenSDK-cached NWB on first run (cached to `outputs/cross_session/<session_id>/`) and then executes A2/B2/D2/H.

Memory budget: a single session at a time. Roughly 3–4 GB peak RAM per run. Total time ~20–40 min per session on an M3 Pro.

In [ ]:
all_results = {}
for sid in SESSIONS:
    all_results[sid] = run_all(sid)
print('\nDone. Sessions complete:', list(all_results.keys()))

## Step 2 — Aggregate results across sessions

In [ ]:
# Cross-session summary tables
def stack(analysis):
    dfs = []
    for sid, res in all_results.items():
        a = res.get(analysis)
        if a is None or 'area_df' not in a:
            continue
        df = a['area_df'].copy()
        df['session_id'] = sid
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

stacked_A = stack('A2')
stacked_B = stack('B2')
stacked_D = stack('D2')
stacked_H = stack('H')

print('A2 rows:', len(stacked_A))
print('B2 rows:', len(stacked_B))
print('D2 rows:', len(stacked_D))
print('H rows:', len(stacked_H))

In [ ]:
# Pivot to cross-session matrix (area × session) for each key metric
def pivot_metric(df, metric):
    if df.empty or metric not in df.columns:
        return pd.DataFrame()
    return df.pivot_table(index='area', columns='session_id', values=metric, aggfunc='first')

pivot_A_mi = pivot_metric(stacked_A, 'mi_mean')
pivot_B_miss = pivot_metric(stacked_B, 'ratio_miss')
pivot_D_r2 = pivot_metric(stacked_D, 'full_r2_mean')
pivot_H_dnc = pivot_metric(stacked_H, 'delta_nc')

print('=== A2: active-vs-passive MI (rows=area, cols=session) ===')
print(pivot_A_mi.round(3))
print('\n=== B2: miss-only change amplification ===')
print(pivot_B_miss.round(2))
print('\n=== D2: full encoding R² ===')
print(pivot_D_r2.round(4))
print('\n=== H: Δ noise correlation (active − passive) ===')
print(pivot_H_dnc.round(4))

In [ ]:
# Save cross-session tables
pivot_A_mi.to_csv(CROSS_REPORT_DIR / 'A_mi_cross_session.csv')
pivot_B_miss.to_csv(CROSS_REPORT_DIR / 'B_miss_ratio_cross_session.csv')
pivot_D_r2.to_csv(CROSS_REPORT_DIR / 'D_fullr2_cross_session.csv')
pivot_H_dnc.to_csv(CROSS_REPORT_DIR / 'H_deltaNC_cross_session.csv')
print('Saved cross-session CSVs to', CROSS_REPORT_DIR)

## Step 3 — Cross-session convergence figure

For each of the four analyses, plot the per-area metric across sessions. We want to see whether findings CONVERGE (same sign, similar magnitude) or DIVERGE (single-session quirks).

In [ ]:
ORDER = ['LGd', 'VISp', 'VISl', 'VISal', 'VISrl', 'VISpm', 'VISam',
         'CA1', 'DG', 'ProS', 'SCig', 'MRN']

def _reorder(df):
    areas = [a for a in ORDER if a in df.index]
    return df.reindex(areas)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, (title, pivot, ref_y) in zip(
    axes.flatten(),
    [
        ('A2 — Active vs Passive MI (per session)', pivot_A_mi, 0),
        ('B2 — Miss-only change amplification (per session)', pivot_B_miss, 1),
        ('D2 — Full encoding R² (per session)', pivot_D_r2, 0),
        ('H — Δ noise correlation active − passive (per session)', pivot_H_dnc, 0),
    ],
):
    p = _reorder(pivot) if not pivot.empty else pivot
    if p.empty:
        ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        continue
    for sid in p.columns:
        ax.plot(p.index, p[sid], '-o', label=f'session {sid}', alpha=0.75)
    ax.axhline(ref_y, color='k', lw=0.5, ls='--', alpha=0.5)
    ax.set_xticklabels(p.index, rotation=35, ha='right')
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.suptitle(f'Cross-session replication — N={len(SESSIONS)} sessions', fontsize=12)
fig.tight_layout()
out = CROSS_REPORT_DIR / 'cross_session_convergence.png'
fig.savefig(out, dpi=140, bbox_inches='tight')
print('Saved:', out)
plt.show()

## Step 4 — Convergence report

For each major finding, check sign consistency and magnitude agreement across the N=3 sessions.

In [ ]:
def sign_convergence(pivot, sign_fn=lambda x: np.sign(x)):
    """Per area: fraction of sessions with same sign as mean across sessions."""
    if pivot.empty:
        return pd.DataFrame()
    mean = pivot.mean(axis=1)
    # How many sessions have same sign as the mean?
    conv = pivot.apply(lambda col: sign_fn(col) == sign_fn(mean), axis=0)
    frac = conv.mean(axis=1)
    return pd.DataFrame({'mean_across_sessions': mean, 'frac_sessions_same_sign': frac})

print('=== A2 convergence ===')
print(_reorder(sign_convergence(pivot_A_mi)).round(3))
print('\n=== B2 miss ratio (> 1 = amplified) ===')
# reframe ratio as (ratio - 1) so sign=positive means amplification
print(_reorder(sign_convergence(pivot_B_miss - 1)).round(3))
print('\n=== D2 full R² ===')
print(_reorder(sign_convergence(pivot_D_r2)).round(3))
print('\n=== H Δ noise correlation ===')
print(_reorder(sign_convergence(pivot_H_dnc)).round(3))

In [ ]:
# Specific headline-finding checks
headline = {}

# 1) SCig active-dominance: should see positive A2 MI across sessions, positive D2 R²
if 'SCig' in pivot_A_mi.index:
    headline['SCig_A2_MI'] = pivot_A_mi.loc['SCig'].to_dict()
if 'SCig' in pivot_D_r2.index:
    headline['SCig_D2_fullR2'] = pivot_D_r2.loc['SCig'].to_dict()

# 2) Hippocampus reward-driven: DG/CA1 miss ratio should be < 1 (collapses)
for a in ['DG', 'CA1']:
    if a in pivot_B_miss.index:
        headline[f'{a}_B2_miss_ratio'] = pivot_B_miss.loc[a].to_dict()

# 3) Noise correlation restructuring: visual cortex positive Δ, MRN negative
for a in ['LGd', 'VISp', 'VISl', 'VISal', 'VISpm', 'VISam', 'SCig', 'MRN']:
    if a in pivot_H_dnc.index:
        headline[f'{a}_H_deltaNC'] = pivot_H_dnc.loc[a].to_dict()

import json
(CROSS_REPORT_DIR / 'headline_replication.json').write_text(json.dumps(headline, indent=2, default=str))
print('Saved headline replication JSON')
print(json.dumps(headline, indent=2, default=str)[:3000])